In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install gymnasium stable-baselines3 sb3-contrib shimmy

import sys
import os

PROJECT_PATH = '/content/drive/MyDrive/MusProject'
sys.path.append(PROJECT_PATH)

try:
    import mus_gym
    print("Librería mus_gym cargada correctamente desde Drive.")
except ImportError:
    print("Error: No se encuentra mus_gym. Revisa la ruta PROJECT_PATH.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Librería mus_gym cargada correctamente desde Drive.


In [ ]:
import gymnasium as gym
import numpy as np
import os
import random
from datetime import datetime
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback, BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from sb3_contrib import RecurrentPPO

import mus_gym
from mus_gym.mus_env import MusEnvironment
from mus_gym.mus_bots import HeuristicBot
from mus_gym.mus_enums import MusAction

from sb3_contrib import RecurrentPPO
from mus_gym import MusEnvConBots

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/MusProject"
MODELS_DIR = os.path.join(DRIVE_ROOT, "models_colab_v4")
LOGS_DIR = os.path.join(DRIVE_ROOT, "logs_colab_v4")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

In [ ]:
NUM_ENVS = 16  
TOTAL_TIMESTEPS = 8_000_000 
PHASE_1_STEPS = 1000000   
CHECKPOINT_FREQ = 250_000
LOAD_MODEL = True
MODEL_PATH_TO_RESUME = os.path.join(MODELS_DIR, "latest_model_colab")

In [ ]:
class SheriffBot(HeuristicBot):
    """
    Un bot que NO se retira si tiene una mano decente.
    Su único propósito es castigar los Órdagos aleatorios de la IA.
    """
    def __init__(self, player_id=1):
        super().__init__(player_id=player_id, prob_farol=0.05, prob_error=0.02)

    def choose_action(self, observation):
        legal = observation["legal_actions_enums"]
        if not legal: return None

        state = observation["game_state"]
        hand_info = observation["my_hand_info"]
        lance = state["current_lance"]

        fuerza = self._calcular_fuerza(lance, hand_info)

        if state["bet"] >= 30: 
            if fuerza > 0.40 and MusAction.QUIERO in legal:
                return MusAction.QUIERO
            if MusAction.NO_QUIERO in legal:
                return MusAction.NO_QUIERO

        return super().choose_action(observation)

class SheriffEnv(MusEnvironment):
    def __init__(self):
        super().__init__(target_score=30)
        self.bot = SheriffBot(player_id=1)
        self.oponente_actual = self.bot 

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        while self.current_player_idx == 1 and not self.game_over:
            obs_dict = self._get_obs_dict_for_bot(1)
            act = self.bot.choose_action(obs_dict)
            if act is None: break
            obs, r, term, trunc, info = super().step(act.value)
            if term or trunc: break
        return obs, info

    def step(self, action):
        o, r, d, t, i = super().step(action)
        if d or t: return o, r, d, t, i

        reward_accum = 0
        while self.current_player_idx == 1 and not (d or t):
            obs_dict = self._get_obs_dict_for_bot(1)
            act = self.bot.choose_action(obs_dict)
            if act is None: break
            o, r_bot, d, t, i = super().step(act.value)
            reward_accum -= r_bot 

        return o, r + reward_accum, d, t, i

    def _get_obs_dict_for_bot(self, p_idx):
        return super()._get_obs_dict_for_bot(p_idx) if hasattr(super(), '_get_obs_dict_for_bot') else {}

from mus_gym import MusEnvConBots
class SheriffEnvWrapper(MusEnvConBots):
    def __init__(self):
        super().__init__()
        self.bot_oponente = SheriffBot(player_id=1)
        self.oponente_actual = self.bot_oponente

class PhaseChangeCallback(BaseCallback):
    def __init__(self, switch_step, save_path, verbose=1):
        super().__init__(verbose)
        self.switch_step = switch_step
        self.save_path = save_path
        self.switched = False

    def _on_step(self) -> bool:
        if not self.switched and self.num_timesteps >= self.switch_step:
            print(f"\n\n--- FASE 1 COMPLETADA ({self.num_timesteps} pasos) ---")
            print("--- Guardando modelo 'Disciplinado' y cambiando estrategia... ---")
            self.model.save(f"{self.save_path}/model_phase1_disciplined")

            self.switched = True
            return False 
        return True



In [ ]:
def main():
    print(f"--- 🚀 INICIANDO ENTRENAMIENTO EN COLAB PRO+ ---")
    print(f"--- Guardando en: {MODELS_DIR} ---")

    if not LOAD_MODEL:
        print("\n--- [FASE 1] ENTRENANDO CONTRA SHERIFF BOT (DISCIPLINA) ---")
        env_phase1 = make_vec_env(SheriffEnvWrapper, n_envs=NUM_ENVS, vec_env_cls=SubprocVecEnv)

        model = RecurrentPPO(
            "MlpLstmPolicy",
            env_phase1,
            verbose=1,
            tensorboard_log=LOGS_DIR,
            learning_rate=0.0003,
            n_steps=512, 
            batch_size=512,
            gamma=0.995,
            ent_coef=0.03, 
            policy_kwargs=dict(lstm_hidden_size=256, net_arch=dict(pi=[256, 256], vf=[256, 256]))
        )

        checkpoint_cb = CheckpointCallback(save_freq=CHECKPOINT_FREQ // NUM_ENVS, save_path=MODELS_DIR, name_prefix="phase1_sheriff")
        phase_cb = PhaseChangeCallback(PHASE_1_STEPS, MODELS_DIR)

        model.learn(total_timesteps=PHASE_1_STEPS + 50000, callback=[checkpoint_cb, phase_cb])
        env_phase1.close()

        path_phase1 = f"{MODELS_DIR}/model_phase1_disciplined.zip"
        print(f"Modelo Fase 1 guardado en {path_phase1}")
        model = RecurrentPPO.load(path_phase1)
    else:
        print(f"Reanudando desde modelo: {MODEL_PATH_TO_RESUME}")
        model = RecurrentPPO.load(MODEL_PATH_TO_RESUME)

    print("\n--- [FASE 2] ENTRENAMIENTO SELF-PLAY (LEAGUE) ---")

    from mus_gym import MusEnvSelfPlay

    initial_opponent_path = os.path.join(MODELS_DIR, "opponent_initial.zip")
    model.save(initial_opponent_path)

    env_kwargs = {"opponent_path": initial_opponent_path}
    env_selfplay = make_vec_env(MusEnvSelfPlay, n_envs=NUM_ENVS, vec_env_cls=SubprocVecEnv, env_kwargs=env_kwargs)

    model.set_env(env_selfplay)

    class DriveLeagueCallback(BaseCallback):
        def __init__(self, update_freq=50000, verbose=1):
            super().__init__(verbose)
            self.update_freq = update_freq
            self.last_update = 0

        def _on_step(self):
            if self.num_timesteps - self.last_update >= self.update_freq:
                self.last_update = self.num_timesteps
                timestamp = datetime.now().strftime("%H%M")
                new_champ_path = os.path.join(MODELS_DIR, f"champion_{timestamp}.zip")

                print(f"[League] 🔄 Actualizando oponente en paso {self.num_timesteps}")
                self.model.save(new_champ_path)

                self.training_env.env_method("update_opponent_model", new_champ_path)

                self.model.save(os.path.join(MODELS_DIR, "latest_model_colab"))
            return True

    model.learn(
        total_timesteps=TOTAL_TIMESTEPS - PHASE_1_STEPS,
        callback=DriveLeagueCallback(update_freq=100_000),
        tb_log_name="PPO_SelfPlay_Colab"
    )

    final_path = os.path.join(MODELS_DIR, "FINAL_MUS_MODEL.zip")
    model.save(final_path)
    print(f"✅ ENTRENAMIENTO COMPLETADO. Modelo final en: {final_path}")
    env_selfplay.close()

if __name__ == "__main__":
    main()

Se han truncado las últimas 5000 líneas del flujo de salida.
|    value_loss           | 0.355      |
----------------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 11.1       |
|    ep_rew_mean          | 0.494      |
| time/                   |            |
|    fps                  | 357        |
|    iterations           | 619        |
|    time_elapsed         | 14176      |
|    total_timesteps      | 5070848    |
| train/                  |            |
|    approx_kl            | 0.13742083 |
|    clip_fraction        | 0.247      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.383     |
|    explained_variance   | -0.0622    |
|    learning_rate        | 0.0003     |
|    loss                 | 0.0982     |
|    n_updates            | 10700      |
|    policy_gradient_loss | -0.0469    |
|    value_loss           | 0.366      |
-------------------------------------

In [ ]:


LATEST_MODEL = os.path.join(MODELS_DIR, "latest_model_colab.zip")

class SuperSheriffBot(HeuristicBot):
    def __init__(self, player_id=1):
        super().__init__(player_id=player_id, prob_farol=0.0, prob_error=0.0) 

    def choose_action(self, observation):
        legal = observation["legal_actions_enums"]
        if not legal: return None

        state = observation["game_state"]
        hand_info = observation["my_hand_info"]
        lance = state["current_lance"]
        fuerza = self._calcular_fuerza(lance, hand_info)

        if state["bet"] >= 30:
            if fuerza > 0.25 and MusAction.QUIERO in legal:
                return MusAction.QUIERO
            if MusAction.NO_QUIERO in legal:
                return MusAction.NO_QUIERO

        return super().choose_action(observation)

class ReformSchoolEnv(MusEnvConBots):
    def __init__(self):
        super().__init__()
        self.bot_oponente = SuperSheriffBot(player_id=1)
        self.oponente_actual = self.bot_oponente

def run_correction():
    print("--- INICIANDO TERAPIA DE CHOQUE (REFORMATORIO) ---")
    print(f"Cargando al delincuente desde: {LATEST_MODEL}")

    env = make_vec_env(ReformSchoolEnv, n_envs=16, vec_env_cls=SubprocVecEnv)

    try:
        model = RecurrentPPO.load(LATEST_MODEL, env=env)
    except Exception as e:
        print(f"Error cargando modelo: {e}")
        return

    model.ent_coef = 0.01

    print("Entrenando 1,000,000 pasos contra Super Sheriff...")
    model.learn(total_timesteps=1_800_000, tb_log_name="PPO_Correction_Therapy")

    rehabilitated_path = os.path.join(MODELS_DIR, "latest_model_colab.zip") 
    model.save(rehabilitated_path)
    print(f" REHABILITACIÓN COMPLETADA. Modelo guardado en: {rehabilitated_path}")
    print("Ahora puedes volver a ejecutar el script de Self-Play (Fase 2).")
    env.close()

if __name__ == "__main__":
    run_correction()